# Homework: Two Energies from One Classifier
**SEAS 8525 · Week 11 · Energy-Based Models · Dr. Mohamed Elbasheer**

In this notebook you take an ordinary MNIST classifier and read it as an energy-based model (EBM) in two ways:

| Reading | Formula | Question it answers |
|---|---|---|
| Pair energy | $E(x, y) = -z_y$ | Does label $y$ fit image $x$? |
| Image energy | $E(x) = -\log \sum_{y=0}^{9} e^{z_y}$ | Does $x$ look like a digit at all? |

Then you fine-tune the classifier the JEM way (Grathwohl et al., ICLR 2020) and see what changes.

**No coding is required.** Run the cells in order. Where you see a settings form, change the values and run the cell again. Your answers go in your reflection, not in this notebook.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.** The notebook runs on a CPU too, but parts 3 and 4 will be slow.
2. **Runtime → Run all** once to check that everything works (about 10 minutes on a GPU).
3. Then go back to the top and work through each part, answering the questions as you go.
4. When you finish, **File → Download → .ipynb**, with all outputs visible.

### The five parts
1. Train a classifier and read its logits as energies (check one image by hand)
2. Use E(x) to tell real digits from other images
3. Langevin sampling with the plain classifier
4. JEM-style fine-tune, then repeat parts 2 and 3
5. Bolt an EBM onto your own project (a one-page design, no notebook work)

## Setup
Run this cell once. It loads PyTorch, downloads MNIST and Fashion-MNIST, and defines a few helpers.

In [ ]:
#@title Setup (run once)
import copy, time, math
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Pixels scaled to [-1, 1], the same range Langevin will clip to.
tf = T.Compose([T.ToTensor(), T.Lambda(lambda x: x * 2 - 1)])
mnist_train = torchvision.datasets.MNIST("data", train=True, download=True, transform=tf)
mnist_test = torchvision.datasets.MNIST("data", train=False, download=True, transform=tf)
fashion_test = torchvision.datasets.FashionMNIST("data", train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=64, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=500)

def show_grid(imgs, nrow, title=None, scale=1.1):
    """imgs: tensor (N, 1, 28, 28) in [-1, 1]."""
    imgs = imgs.detach().cpu().clamp(-1, 1)
    n = imgs.shape[0]; ncol = math.ceil(n / nrow)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * scale, nrow * scale))
    axes = np.array(axes).reshape(nrow, ncol)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n: ax.imshow(imgs[i, 0], cmap="gray", vmin=-1, vmax=1)
    if title: fig.suptitle(title)
    plt.tight_layout(); plt.show()

print("Train images:", len(mnist_train), "| Test images:", len(mnist_test))

---
## Part 1: Read a classifier's logits as energies

### 1a. Train an ordinary classifier
A small CNN: four convolution layers, then two fully connected layers that end in **10 logits**, $z_0$ to $z_9$.
It is trained the normal way, with cross-entropy. Nothing about it is special to EBMs.

(It uses no batch normalization, because Langevin sampling later needs the network to treat every image on its own.)

In [ ]:
#@title 1a. Train the classifier (about 1 minute on a GPU)
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.SiLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.SiLU(),      # 28x28 -> 14x14
            nn.Conv2d(64, 64, 3, 1, 1), nn.SiLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.SiLU(),     # 14x14 -> 7x7
            nn.Flatten(), nn.Linear(128 * 7 * 7, 128), nn.SiLU(),
            nn.Linear(128, 10))                         # 10 logits
    def forward(self, x):
        return self.body(x)

def energies(model, x):
    """Returns (E(x, y) for all 10 labels, E(x))."""
    z = model(x)
    return -z, -torch.logsumexp(z, dim=1)

@torch.no_grad()
def accuracy(model):
    model.eval(); right = 0
    for x, y in test_loader:
        right += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
    return right / len(mnist_test)

clf = Net().to(device)
opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(2):
    clf.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = F.cross_entropy(clf(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch + 1}: test accuracy {accuracy(clf):.2%}  ({time.time() - t0:.0f}s)")
clf_state = copy.deepcopy(clf.state_dict())   # saved so part 4 starts from this exact classifier

### 1b. One image, both energies, by hand
The next cell takes one test image and prints every number you need. Change `image_index` to look at other images.

In [ ]:
#@title 1b. Show the two energies for one image
image_index = 0 #@param {type:"integer"}

clf.eval()
x, y_true = mnist_test[image_index]
with torch.no_grad():
    z = clf(x[None].to(device))[0].cpu().double()
ez = torch.exp(z); S = ez.sum()
plt.figure(figsize=(1.6, 1.6)); plt.imshow(x[0], cmap="gray"); plt.axis("off"); plt.title(f"true label: {y_true}"); plt.show()

print(f"{'label y':>7} | {'logit z_y':>9} | {'e^(z_y)':>12} | {'E(x, y) = -z_y':>15} | {'softmax p(y|x)':>14}")
print("-" * 70)
for k in range(10):
    print(f"{k:>7} | {z[k]:>9.3f} | {ez[k]:>12.3f} | {-z[k]:>15.3f} | {ez[k] / S:>14.4f}")
print("-" * 70)
print(f"softmax denominator  S = sum of e^(z_y) = {S:.3f}")
print(f"image energy      E(x) = -log S          = {-torch.log(S):.3f}")
print(f"lowest pair energy: label {int((-z).argmin())}, E(x, y) = {(-z).min():.3f}")

### Questions for part 1 (answer in your reflection)
1. For `image_index = 0`, copy the table into your reflection. Using a calculator, check two things by hand: the softmax denominator $S = \sum_y e^{z_y}$, and $E(x) = -\log S$. Show your arithmetic.
2. Which label has the lowest pair energy $E(x, y)$? Is it the same label the classifier predicts? Explain why this must always be true.
3. Compare $E(x)$ with the lowest $E(x, y)$. They are usually very close. Explain why, using the idea that E(x) behaves like a "soft minimum" of the ten pair energies.
4. Add 5 to all ten logits in your head. What happens to the softmax probabilities? What happens to $E(x)$? What does this tell you about what ordinary cross-entropy training can and cannot shape?

---
## Part 2: Can E(x) tell real digits from other images?
If $E(x)$ means "does this look like a digit?", then real MNIST digits should get **low** energy, and clothing images (Fashion-MNIST) and pure noise should get **high** energy.

We measure this with the **AUROC**: the chance that a randomly chosen real digit gets a lower energy than a randomly chosen other image. 1.0 is perfect, 0.5 is a coin flip, and below 0.5 means the energy has it backwards.

As a comparison, we also score each image with the classifier's **max softmax probability**, a common "how sure am I?" score.

In [ ]:
#@title 2. Energy histograms and AUROC for the plain classifier
N_EVAL = 2000
mnist_x = torch.stack([mnist_test[i][0] for i in range(N_EVAL)])
fashion_x = torch.stack([fashion_test[i][0] for i in range(N_EVAL)])
noise_x = torch.rand(N_EVAL, 1, 28, 28) * 2 - 1
SETS = {"MNIST (real digits)": mnist_x, "Fashion-MNIST": fashion_x, "Uniform noise": noise_x}

@torch.no_grad()
def scores(model, x):
    model.eval(); Es, Ps = [], []
    for i in range(0, len(x), 500):
        z = model(x[i:i + 500].to(device))
        Es.append(-torch.logsumexp(z, 1).cpu()); Ps.append(F.softmax(z, 1).max(1).values.cpu())
    return torch.cat(Es).numpy(), torch.cat(Ps).numpy()

def ood_report(model, name):
    res = {k: scores(model, v) for k, v in SETS.items()}
    plt.figure(figsize=(7, 3.2))
    for k, (E, _) in res.items():
        plt.hist(E, bins=60, alpha=0.55, label=k, density=True)
    plt.xlabel("E(x)   (low = looks like a digit)"); plt.ylabel("density"); plt.title(f"Image energy E(x): {name}")
    plt.legend(); plt.tight_layout(); plt.show()
    E_in, P_in = res["MNIST (real digits)"]
    table = {}
    for k in ["Fashion-MNIST", "Uniform noise"]:
        E_out, P_out = res[k]
        labels = np.r_[np.ones(len(E_in)), np.zeros(len(E_out))]
        table[k] = (roc_auc_score(labels, -np.r_[E_in, E_out]), roc_auc_score(labels, np.r_[P_in, P_out]))
    print(f"{'real digits vs':<16} | {'AUROC using -E(x)':>18} | {'AUROC using max softmax':>24}")
    print("-" * 64)
    for k, (a, b) in table.items():
        print(f"{k:<16} | {a:>18.3f} | {b:>24.3f}")
    for k, (E, _) in res.items():
        print(f"mean E(x) on {k:<20}: {E.mean():8.2f}")
    return table

auroc_before = ood_report(clf, "plain classifier")

### Questions for part 2
1. Copy the AUROC table into your reflection. Which "other" set does the plain classifier's E(x) handle better, Fashion-MNIST or noise? Which handles it worse?
2. Look at the histograms. Does any set get **lower** energy than the real digits? If so, explain how that can happen when the classifier was never trained to shape E(x).
3. E(x) and max softmax probability are built from the same ten logits. Why can they still rank images differently? (Hint: part 1, question 4.)

---
## Part 3: Langevin sampling with the plain classifier
This is the **inner loop**. We start from pure noise and repeat one step:

$$x \leftarrow x - \eta \, \nabla_x E + \text{noise}, \qquad \text{then clip the pixels to } [-1, 1]$$

- **Conditional:** $E = E(x, y) = -z_y$ with the label $y$ fixed, so we roll toward "looks like a $y$".
- **Unconditional:** $E = E(x)$, so we roll toward "looks like any digit".

The network's **weights are frozen** the whole time. Each step runs the images forward through every layer to get the energy, then backpropagates that energy through every layer to the **pixels**. Only the pixels change.

The cell shows 8 samples (rows) at several moments of the loop (columns), the average energy at every step, and what the classifier itself predicts for the final images.

In [ ]:
#@title 3. Langevin settings (change these and run the cell again)
mode = "conditional E(x, y)" #@param ["conditional E(x, y)", "unconditional E(x)"]
target_label = 7 #@param {type:"slider", min:0, max:9, step:1}
n_steps = 60 #@param {type:"slider", min:0, max:200, step:10}
step_size = 1.0 #@param {type:"number"}
noise_std = 0.01 #@param {type:"number"}

def langevin(model, x, n_steps, step_size, noise_std, y=None, snap_every=10):
    """Inner loop. Changes the pixels of x; the model's weights stay frozen."""
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)          # weights frozen
    x = x.clone().detach(); snaps = [x.cpu()]; trace = []
    for t in range(n_steps):
        x.requires_grad_(True)
        E_xy, E_x = energies(model, x)                             # forward through every layer
        E = E_xy.gather(1, y[:, None]).squeeze(1) if y is not None else E_x
        grad = torch.autograd.grad(E.sum(), x)[0]                  # backward through every layer, to the pixels
        x = (x - step_size * grad + noise_std * torch.randn_like(x)).clamp(-1, 1).detach()
        trace.append(E.mean().item())
        if (t + 1) % snap_every == 0: snaps.append(x.cpu())
    for p in model.parameters(): p.requires_grad_(True)
    return x, snaps, trace

def langevin_demo(model, name, n=8):
    torch.manual_seed(1)
    x0 = (torch.rand(n, 1, 28, 28) * 2 - 1).to(device)
    y = torch.full((n,), target_label, device=device) if mode.startswith("conditional") else None
    x_final, snaps, trace = langevin(model, x0, n_steps, step_size, noise_std, y=y)
    grid = torch.stack(snaps, dim=1).reshape(-1, 1, 28, 28)      # rows = samples, columns = steps 0, 10, 20, ...
    show_grid(grid, nrow=n, title=f"{name}: {mode}" + (f", y = {target_label}" if y is not None else "") + f"  (columns = every 10 steps, left = noise)", scale=0.9)
    if trace:
        plt.figure(figsize=(5, 2.6)); plt.plot(trace); plt.xlabel("Langevin step"); plt.ylabel("mean energy")
        plt.title("Energy along the inner loop"); plt.tight_layout(); plt.show()
    with torch.no_grad():
        p = F.softmax(model(x_final), 1)
    print("Classifier's prediction for each final sample:")
    for i in range(n):
        print(f"  sample {i + 1}: predicts {int(p[i].argmax())} with probability {p[i].max().item():.1%}")
    return x_final

samples_before = langevin_demo(clf, "Plain classifier")

### Questions for part 3
Run the cell at least four times and save the image strips you discuss:
1. **Default settings** (conditional, y = 7, 60 steps). Do the samples look like 7s to you? What does the classifier predict for them, and how confident is it? What does the gap between your eyes and its confidence tell you?
2. **Step size.** Try 0.1 and 10. Describe what changes, both in the images and in the energy curve.
3. **Number of steps.** Try 10 and 200. How many steps does the energy need before it levels off?
4. **Unconditional E(x).** Switch the mode. How do these samples compare with the conditional ones?
5. In your own words: during this cell, which numbers changed, and which numbers did not change at all?

---
## Part 4: JEM-style fine-tune, then compare
Now we teach the network to shape E(x) too. Each training round has two loops (the same picture as the handout):

- **Inner loop (Langevin, weights frozen):** make negatives. Start from a replay buffer of old negatives (95%) or fresh noise (5%), and run 20 Langevin steps on $E(x)$.
- **Outer loop (training, weights updated):** score the positives (real digits) and the negatives with the same network.

$$\mathcal{L} = \mathrm{CE} + \Big[\, \overline{E(x_{\text{pos}})} - \overline{E(x_{\text{neg}})} \,\Big] + \alpha \Big(\, \overline{E(x_{\text{pos}})^2} + \overline{E(x_{\text{neg}})^2} \,\Big)$$

A bar over a term means its average over the batch.

The cross-entropy keeps the classifier working. The middle term pushes the positive energies down and the negative energies up. The $\alpha$ term keeps the numbers from blowing up.

EBM training can be unstable. If the energies explode, the cell stops on its own and keeps the last healthy checkpoint.

In [ ]:
#@title 4a. Fine-tune (about 3 to 8 minutes on a GPU)
n_iters = 1500 #@param {type:"integer"}
langevin_steps_train = 20 #@param {type:"integer"}
alpha = 0.01 #@param {type:"number"}

jem = Net().to(device); jem.load_state_dict(clf_state)            # start from the exact classifier of part 1
opt = torch.optim.Adam(jem.parameters(), lr=1e-4)
buffer = torch.rand(10000, 1, 28, 28) * 2 - 1                      # replay buffer of negatives
history = {"it": [], "E_pos": [], "E_neg": [], "acc": []}
good_state = copy.deepcopy(jem.state_dict())
batches = iter(train_loader); t0 = time.time()

for it in range(1, n_iters + 1):
    try: x_pos, y_pos = next(batches)
    except StopIteration: batches = iter(train_loader); x_pos, y_pos = next(batches)
    x_pos, y_pos = x_pos.to(device), y_pos.to(device)

    # ---- inner loop: make the negatives (weights frozen) ----
    idx = torch.randint(0, len(buffer), (x_pos.shape[0],))
    x0 = buffer[idx].clone()
    fresh = torch.rand(len(idx)) < 0.05
    x0[fresh] = torch.rand(int(fresh.sum()), 1, 28, 28) * 2 - 1
    x_neg, _, _ = langevin(jem, x0.to(device), langevin_steps_train, 1.0, 0.01, snap_every=10**9)
    buffer[idx] = x_neg.cpu()

    # ---- outer loop: score positives and negatives, update the weights ----
    jem.train()
    logits_pos = jem(x_pos)
    E_pos = -torch.logsumexp(logits_pos, 1)
    E_neg = -torch.logsumexp(jem(x_neg), 1)
    loss = F.cross_entropy(logits_pos, y_pos) + (E_pos.mean() - E_neg.mean()) + alpha * (E_pos.pow(2).mean() + E_neg.pow(2).mean())
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(jem.parameters(), 10.0)
    opt.step()

    if not torch.isfinite(loss) or E_pos.abs().mean() > 1e4 or E_neg.abs().mean() > 1e4:
        print(f"Energies blew up at round {it}. Restoring the last healthy checkpoint and stopping.")
        jem.load_state_dict(good_state); break
    if it % 100 == 0:
        acc = accuracy(jem); good_state = copy.deepcopy(jem.state_dict())
        history["it"].append(it); history["E_pos"].append(E_pos.mean().item()); history["E_neg"].append(E_neg.mean().item()); history["acc"].append(acc)
        print(f"round {it:5d} | E(pos) {E_pos.mean().item():8.2f} | E(neg) {E_neg.mean().item():8.2f} | test accuracy {acc:.2%} | {time.time() - t0:.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history["it"], history["E_pos"], label="E(positives)"); ax[0].plot(history["it"], history["E_neg"], label="E(negatives)")
ax[0].set_xlabel("training round"); ax[0].set_title("Energies during training"); ax[0].legend()
ax[1].plot(history["it"], history["acc"]); ax[1].set_xlabel("training round"); ax[1].set_title("Test accuracy")
plt.tight_layout(); plt.show()
show_grid(buffer[torch.randperm(len(buffer))[:32]], nrow=4, title="32 negatives from the replay buffer", scale=0.9)

In [ ]:
#@title 4b. Repeat part 2 with the fine-tuned network
auroc_after = ood_report(jem, "after JEM-style fine-tune")
print(f"\nTest accuracy: plain classifier {accuracy(clf):.2%}  ->  fine-tuned {accuracy(jem):.2%}")
print(f"\n{'real digits vs':<16} | {'AUROC -E(x) before':>18} | {'AUROC -E(x) after':>18}")
for k in auroc_before:
    print(f"{k:<16} | {auroc_before[k][0]:>18.3f} | {auroc_after[k][0]:>18.3f}")

In [ ]:
#@title 4c. Repeat part 3 with the fine-tuned network (uses the settings form from part 3)
samples_before = langevin_demo(clf, "Plain classifier")     # rerun so both use the current settings
samples_after = langevin_demo(jem, "Fine-tuned network")
show_grid(torch.cat([samples_before[:8].cpu(), samples_after[:8].cpu()]), nrow=2,
          title="Top row: plain classifier.  Bottom row: fine-tuned.  Same noise, same settings.", scale=1.0)

### Questions for part 4
1. Describe the energies-during-training plot. Do E(positives) and E(negatives) move the way the loss says they should? Does the gap stay steady or swing back and forth?
2. Did test accuracy change? Is the change worth what you gained? Use your numbers.
3. Fill in a before and after table: test accuracy, AUROC against Fashion-MNIST, AUROC against noise. Which number moved the most, and why?
4. Put the two sample rows side by side (cell 4c, with y = 7). Describe the difference. Then try the unconditional mode for both networks.
5. Explain in your own words why fine-tuning on $E(x)$ changed the conditional samples from $E(x, y)$, even though the loss never used $E(x, y)$ directly. (Hint: both energies come from the same logits.)
6. Count the cost. Each training round ran 20 Langevin steps. Roughly how many forward and backward passes did the whole fine-tune take, compared with one epoch of ordinary training?

---
## Part 5: Bolt an EBM onto your own project (one page, no notebook work)
Write one page, inside your reflection document, that answers these four questions for the research project you are proposing this semester:
1. **The energy.** Which network do you already use? Would it become $E(x)$ or $E(x, y)$, and what are $x$ and $y$ in your problem? What is the single output?
2. **Positives and negatives.** Where do your positives come from? How would you make negatives? If your data is discrete (text, graphs, tokens), Langevin does not apply directly. Say what you would use instead.
3. **One ablation.** Baseline, versus baseline plus EBM, measured by which metric on which dataset.
4. **The biggest risk** (cost, instability, discrete data, something else) and your fallback if it happens.

---
## What to submit
1. This notebook, downloaded as **.ipynb with all outputs visible**.
2. A reflection (PDF or Word): answers to the questions in parts 1 to 4 (**700 to 1,000 words**, plus your tables and image strips), and the **one-page design** from part 5.